## Comparing Movie Recommendation Ranking methods

Here we experiment whether ranking movies based on difference between a user's rating habits/predicted ratings and a movie's average rating. We question whether the relationship between these two variables will produce more personalized reccomendation over ranking movies by their predicted ratings.

So here we compare: Standard Ranking and Difference-based ranking where Standard ranking ranks movies from their highest to lowest predicted rating and Difference-based ranking is where movies are ranked by the user's rating habits/predicted ratings compares to the movie's average rating. Then we examine the number of ratings ech recommended movie has recieved, this is to determine which method favors more popular movies.


In [2]:

import pandas as pd
from pathlib import Path
from surprise import Dataset, Reader, SVD

# Load the Letterboxd dataset
path = Path("../reviews_filtered.csv")

if not path.exists():
    path = Path("reviews_filtered.csv")

data = pd.read_csv(path)

print("Total ratings:", len(data)) # i pushed the changes w the preciison and recall (idk why i thought we needed that) but it also has rmse and mse, but both of those are pretty low?? they're like 1.2 and 2.sth (respectively) so it's not bad. but we could def optimize a little more 

print("Unique movies:", data["movie_id"].nunique())
print("Unique users:", data["user_id"].nunique())

# Load ratings into Surprise
reader = Reader(rating_scale=(1, 10))

dataset = Dataset.load_from_df(
    data[["user_id", "movie_id", "rating"]],
    reader
)

# Train SVD model
model = SVD(random_state=42)
model.fit(dataset.build_full_trainset())


Total ratings: 855434
Unique movies: 5937
Unique users: 2516


In [3]:
# Select an existing user rating history
user_counts = data["user_id"].value_counts()
user_id = user_counts.index[min(100, len(user_counts) - 1)]

# Find movies user has already rated
rated_movies = set(
    data.loc[data["user_id"] == user_id, "movie_id"]
)

# Calculate average ratings and rating counts for each movie
movie_stats = (
    data.groupby("movie_id")
    .agg(
        name=("name", "first"),
        avg_rating=("rating", "mean"),
        rating_count=("rating", "size"))
    .reset_index()
)

# Keep movies user hasn't rated and discard already rated movies
candidates = movie_stats[ ~movie_stats["movie_id"].isin(rated_movies)].copy()

# Predict the user's rating for each unrated movie
candidates["predicted_rating"] = [
    model.predict(user_id, movie_id).est
    for movie_id in candidates["movie_id"]
]

# Calculate the difference between predicted and average rating
candidates["difference"] = (
    candidates["predicted_rating"] - candidates["avg_rating"]
)

#showing some comparison
print("Selected user:", user_id)
print("Movies already rated:", len(rated_movies))
print("Movies available for recommendations:", len(candidates))

display(candidates.head(10))


Selected user: 538143
Movies already rated: 1203
Movies available for recommendations: 4734


,movie_id,name,avg_rating,rating_count,predicted_rating,difference
0,247,take-shelter,7.918605,86,6.198003,-1.720601
2,278,killing-them-softly,6.710843,83,7.150416,0.439572
4,281,47-ronin-2013,4.029412,34,5.539849,1.510437
5,282,extremely-loud-incredibly-close,4.763158,38,7.012602,2.249444
7,331,total-recall-2012,4.717391,46,6.020737,1.303345
8,603,the-muppets,7.353846,130,7.059455,-0.294392
9,994,jackass-35,7.108108,37,6.548498,-0.559610
10,1081,happy-feet-two,4.341463,41,4.960940,0.619477
11,1086,the-girl-with-the-dragon-tattoo-2011,7.791855,221,8.238587,0.446732
12,1599,tomboy-2011,7.910448,67,7.648236,-0.262212


Process in the cell:
1. select a user that has a good history in rating movies
2. remove all the movies they already rated
3. use the SVD model we trained earler to predict how a user would rate the remaining movies
4. Calculate the difference between movies average rating from their predicted rating.

In [4]:
#sorting based on highest recommendation score
# i think these values displayed might indicate that the model is just recommending movies based on their popularity
# as the scale of the predicted ratings is very similar to the average ratings



top_by_prediction = candidates.sort_values(
    "predicted_rating", ascending=False
).head(10)

mid_by_prediction = candidates.sort_values(
    "predicted_rating", ascending=False
).head(candidates.shape[0]//2).tail(10)

bottom_by_prediction = candidates.sort_values(
    "predicted_rating", ascending=True
).head(10)

print("Top 10 recommended movies:")
display(top_by_prediction[
    ["name", "predicted_rating", "avg_rating", "rating_count", "difference"]
])

print("Middle 10 recommended movies:")
display(mid_by_prediction[
    ["name", "predicted_rating", "avg_rating", "rating_count", "difference"]
])

print("Bottom 10 recommended movies:")
display(bottom_by_prediction[
    ["name", "predicted_rating", "avg_rating", "rating_count", "difference"]
])

Top 10 recommended movies:


,name,predicted_rating,avg_rating,rating_count,difference
5286,all-too-well-the-short-film,10.000000,7.442478,339,2.557522
5838,taylor-swift-the-eras-tour,9.974788,8.355882,340,1.618905
4089,when-they-see-us,9.846744,9.103448,87,0.743296
1572,the-empire-strikes-back,9.823741,9.093137,204,0.730604
356,inception,9.786956,8.383202,381,1.403753
3288,lemonade-2016,9.705415,8.662162,74,1.043253
2116,amelie,9.646976,8.078313,166,1.568663
4691,8-46-2020,9.644496,8.800000,40,0.844496
2187,before-sunrise,9.630342,8.861607,224,0.768735
4279,and-then-we-danced,9.599637,8.033708,89,1.565929


Middle 10 recommended movies:


,name,predicted_rating,avg_rating,rating_count,difference
566,cant-hardly-wait,6.353873,5.914286,35,0.439587
2136,akira,6.352346,8.266129,124,-1.913783
1637,die-hard-2,6.351688,6.266667,75,0.085022
880,daddy-day-care,6.349920,4.822222,45,1.527698
2804,selma,6.349856,7.412844,109,-1.062988
1260,8mm,6.349641,6.090909,33,0.258732
5500,the-book-of-clarence-2023,6.348461,5.761905,126,0.586556
2510,the-drop-2014,6.347147,6.682540,63,-0.335393
568,rachel-getting-married,6.346926,7.634146,41,-1.287220
1842,tomorrow-never-dies,6.346673,6.000000,68,0.346673


Bottom 10 recommended movies:


,name,predicted_rating,avg_rating,rating_count,difference
5513,winnie-the-pooh-blood-and-honey,1.000000,2.417857,280,-1.417857
4568,365-days-2020,1.000000,1.464789,142,-0.464789
5099,365-days-this-day,1.000000,1.333333,63,-0.333333
2254,the-human-centipede-2-full-sequence,1.000000,3.000000,44,-2.000000
4029,cats-2019,1.000000,2.817927,357,-1.817927
3979,morbius,1.000000,2.901408,568,-1.901408
3873,the-fanatic,1.080255,3.029703,101,-1.949448
5275,the-black-demon,1.174340,3.287879,66,-2.113539
5531,best-christmas-ever-2023,1.495536,2.920635,63,-1.425099
5098,the-next-365-days,1.509680,1.434783,46,0.074898


In [ ]:
from surprise import Dataset, Reader, SVD


def evaluate_fixed_holdouts(
    data,
    holdout_movies_by_user,
    like_threshold=8,
    k=10,
    seed=42,
):
    """
    Hide specified movie ratings per randomly selected sample of users, 
    train SVD on the remaining data,
    and compare predicted-rating and popularity rankings.

    Helper function to the set of selected users. 

    holdout_movies_by_user format:
        {user_id: [movie_id_1, movie_id_2, ...]}
    these movies are randomized for each user. should we keep the list of movies the same across
    all users? I'm thinking that may introduce bias towards the average rating of the movie, 
    and will cause the model to overfit. 

    Returns:
        summary: average precision/recall by ranking method
        per_user_metrics: metric values per user and method
        top_lists: top-k recommendations for each user and method
        heldout_movies: selected holdouts with predictions and training counts
    """
    required = {"user_id", "movie_id", "name", "rating"}
    missing = required - set(data.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    if not isinstance(k, int) or k <= 0:
        raise ValueError("k must be a positive integer.")

    if data.duplicated(["user_id", "movie_id"]).any():
        raise ValueError(
            "Duplicate user/movie rows found. Resolve these before evaluating."
        )

    if not holdout_movies_by_user:
        raise ValueError("Provide at least one user and holdout movie.")

    existing_pairs = set(zip(data["user_id"], data["movie_id"]))
    requested_pairs = set()

    for user_id, movie_ids in holdout_movies_by_user.items():
        if len(movie_ids) != len(set(movie_ids)):
            raise ValueError(f"Duplicate holdout movie IDs for user {user_id}.")

        requested_pairs.update((user_id, movie_id) for movie_id in movie_ids)

    missing_pairs = requested_pairs - existing_pairs
    if missing_pairs:
        raise ValueError(
            "Some requested user/movie pairs are not in the dataset. "
            f"Examples: {list(missing_pairs)[:5]}"
        )

    if not requested_pairs:
        raise ValueError("No holdout movie pairs were provided.")

    # Remove only the specified user/movie ratings from training.
    holdout_mask = [
        (user_id, movie_id) in requested_pairs
        for user_id, movie_id in zip(data["user_id"], data["movie_id"])
    ]
    test_data = data.loc[holdout_mask].copy()
    train_data = data.loc[[not held_out for held_out in holdout_mask]].copy()

    for user_id in holdout_movies_by_user:
        if not (train_data["user_id"] == user_id).any():
            raise ValueError(
                f"User {user_id} has no ratings left for model training."
            )

    # Train SVD using all rows remaining in train_data.
    reader = Reader(rating_scale=(1, 10))
    surprise_data = Dataset.load_from_df(
        train_data[["user_id", "movie_id", "rating"]],
        reader,
    )
    trainset = surprise_data.build_full_trainset()

    model = SVD(random_state=seed)
    model.fit(trainset)

    # Keep movie names from the full catalog, but calculate popularity
    # using training ratings only.
    movie_catalog = data[["movie_id", "name"]].drop_duplicates("movie_id")
    training_movie_stats = (
        train_data.groupby("movie_id")["rating"]
        .agg(avg_rating="mean", training_rating_count="size")
        .reset_index()
    )
    training_mean = train_data["rating"].mean()

    metric_rows = []
    top_list_rows = []
    heldout_rows = []

    for user_id in holdout_movies_by_user:
        user_test = test_data[test_data["user_id"] == user_id]

        relevant_movie_ids = set(
            user_test.loc[
                user_test["rating"] >= like_threshold,
                "movie_id",
            ]
        )

        # A candidate is any movie the user did not rate in training.
        seen_in_training = set(
            train_data.loc[
                train_data["user_id"] == user_id,
                "movie_id",
            ]
        )
        candidates = movie_catalog[
            ~movie_catalog["movie_id"].isin(seen_in_training)
        ].copy()

        candidates["predicted_rating"] = [
            model.predict(user_id, movie_id).est
            for movie_id in candidates["movie_id"]
        ]

        candidates = candidates.merge(
            training_movie_stats,
            on="movie_id",
            how="left",
        )
        candidates["avg_rating"] = candidates["avg_rating"].fillna(training_mean)
        candidates["training_rating_count"] = (
            candidates["training_rating_count"].fillna(0).astype(int)
        )

        # Save every held-out movie with its prediction and training count.
        user_heldout_details = candidates.loc[
            candidates["movie_id"].isin(user_test["movie_id"]),
            [
                "movie_id",
                "name",
                "predicted_rating",
                "avg_rating",
                "training_rating_count",
            ],
        ].merge(
            user_test[["movie_id", "rating"]].rename(
                columns={"rating": "heldout_rating"}
            ),
            on="movie_id",
            how="left",
        )

        user_heldout_details.insert(0, "user_id", user_id)
        user_heldout_details["is_relevant"] = (
            user_heldout_details["heldout_rating"] >= like_threshold
        )
        heldout_rows.append(user_heldout_details)

        # Keep the two ranking methods unchanged.
        rankings = {
            "SVD predicted rating": candidates.sort_values(
                ["predicted_rating", "movie_id"],
                ascending=[False, True],
            ),
            "Popularity": candidates.sort_values(
                ["training_rating_count", "movie_id"],
                ascending=[False, True],
            ),
        }

        heldout_rating_by_movie = user_test.set_index("movie_id")["rating"]

        for method, ranking in rankings.items():
            top_k = ranking.head(k).copy()
            recommended_ids = set(top_k["movie_id"])

            # Save the top-k recommendations for this user and method.
            top_k["heldout_rating"] = top_k["movie_id"].map(
                heldout_rating_by_movie
            )
            top_k["has_heldout_rating"] = top_k["heldout_rating"].notna()
            top_k["is_relevant"] = (
                top_k["heldout_rating"].ge(like_threshold).fillna(False)
            )
            top_k.insert(0, "rank", range(1, len(top_k) + 1))
            top_k.insert(0, "method", method)
            top_k.insert(0, "user_id", user_id)
            top_list_rows.append(top_k)

            # Precision/recall can be computed if this user has relevant
            # held-out movies at the chosen threshold.
            if relevant_movie_ids:
                hits = len(recommended_ids & relevant_movie_ids)

                metric_rows.append({
                    "user_id": user_id,
                    "method": method,
                    "hits_in_top_k": hits,
                    "relevant_heldout_movies": len(relevant_movie_ids),
                    "precision_at_k": hits / len(top_k),
                    "recall_at_k": hits / len(relevant_movie_ids),
                })

    per_user_metrics = pd.DataFrame(metric_rows)
    top_lists = pd.concat(top_list_rows, ignore_index=True)
    heldout_movies = pd.concat(heldout_rows, ignore_index=True)

    if per_user_metrics.empty:
        summary = pd.DataFrame(
            columns=[
                "precision_at_k",
                "recall_at_k",
                "users_evaluated",
            ]
        )
    else:
        summary = (
            per_user_metrics
            .groupby("method")
            .agg(
                precision_at_k=("precision_at_k", "mean"),
                recall_at_k=("recall_at_k", "mean"),
                users_evaluated=("user_id", "nunique"),
            )
            .round(3)
        )

    return summary, per_user_metrics, top_lists, heldout_movies

In [6]:
import numpy as np

NUM_TEST_USERS = 40 #how many users to select for testing
MIN_RATINGS = 50 # minimum number of ratings a user must have (in order to be properly split)
HOLDOUT_FRACTION = 0.20 #percentage of movie ratings to be exempt
SEED = 42

ratings_per_user = data.groupby("user_id").size()
eligible_users = ratings_per_user[
    ratings_per_user >= MIN_RATINGS
].index.to_numpy()

if len(eligible_users) < NUM_TEST_USERS:
    raise ValueError(
        f"Only {len(eligible_users)} users have at least "
        f"{MIN_RATINGS} ratings."
    )

# Select a reproducible set of users.
user_rng = np.random.default_rng(SEED)
test_user_ids = user_rng.choice(
    eligible_users,
    size=NUM_TEST_USERS,
    replace=False,
).tolist()

# For each selected user, select movies they actually rated.
holdout_rng = np.random.default_rng(SEED + 1)
holdout_movies_by_user = {}

for user_id in test_user_ids:
    user_movie_ids = data.loc[
        data["user_id"] == user_id,
        "movie_id",
    ].to_numpy()

    n_holdout = max(
        1,
        round(len(user_movie_ids) * HOLDOUT_FRACTION),
    )

    holdout_movies_by_user[user_id] = holdout_rng.choice(
        user_movie_ids,
        size=n_holdout,
        replace=False,
    ).tolist()

print(f"Selected {len(test_user_ids)} test users.")
print("Example holdouts:", list(holdout_movies_by_user.items())[:2])

Selected 40 test users.
Example holdouts: [(771936, [575043, 272503, 48044, 251872, 64393, 451512, 335384, 354251, 523229, 82488, 558956, 715856, 51896, 895130, 551275, 794063, 952557, 69142, 362977, 937215, 37550, 923207, 307239, 16527, 59281, 93676, 268192, 44580, 131984, 354541, 48140, 401687, 774923, 251937, 575258, 444424, 7846, 361725, 296341, 95113, 309555, 850459, 16525, 556785, 73371, 878341, 387238, 125976, 338883, 695629, 546348, 459585, 16530, 589093, 178123, 777185, 526697, 495969, 370178, 207224, 705221, 174928, 459564, 395495, 788966, 284629, 62780, 527841, 789082, 389270, 617443, 52512, 8084, 503890, 488575, 431888, 50955, 496592, 472465, 86991, 268947, 494166, 475370, 47994, 518787, 51614, 556553, 755564, 793883, 946607, 436873, 468929, 270477, 129662, 866275, 150170, 41512, 438511, 52513, 50621, 197012, 332434, 47469, 515909, 378525, 686389, 605262, 261866, 691815, 558056, 333310, 27809, 81909, 311120]), (77148, [706064, 703943, 866275, 709530, 427970, 835774, 449442,

information evaluation section. the output is for ALL users so if you don't want that part, **don't run the second cell (it's 7 for me right now).** 

precision and recall table: displays precision and recall for the SVD model and popularity. the hist_in_top_k counts how many top 10 movies were exempt from training AND rated at or above the like threshold. precision is hits/recommended number, recall is hits/user's exempt and above like level movies. 

selected held out movies: all the movies exempt from training for that user. the held out value is the actual rating they gave and predicted_rating is the SVD's estimate. 

top recommendations by method: shows the top 10 candidate movies for each ranking method (popularity, SVD). SVD sorts by predicted rating, popularity sorts by rating count. the held out rating value is only given if it was exempt, otherwise it was used in training. 

In [7]:
from IPython.display import display, HTML

summary, per_user_metrics, top_lists, heldout_movies = evaluate_fixed_holdouts(
    data=data,
    holdout_movies_by_user=holdout_movies_by_user,
    like_threshold=8,
    k=10,
    seed=SEED,
)

# Overall comparison of SVD and Popularity.
display(HTML("<h2>Overall precision and recall summary</h2>"))
display(summary.reset_index())

# Rating error uses every held-out rating, not only relevant movies or top-k items.
rating_errors = (
    heldout_movies["predicted_rating"]
    - heldout_movies["heldout_rating"]
)
squared_errors = rating_errors ** 2
user_mse = squared_errors.groupby(heldout_movies["user_id"]).mean()

svd_error_summary = pd.DataFrame([{
    "ratings_evaluated": len(rating_errors),
    "mse": squared_errors.mean(),
    "rmse": np.sqrt(squared_errors.mean()),
    "mean_user_mse": user_mse.mean(),
    "mean_user_rmse": np.sqrt(user_mse).mean(),
}])

display(HTML("<h2>SVD rating error on all held-out ratings</h2>"))
display(svd_error_summary.round(3))

,method,precision_at_k,recall_at_k,users_evaluated
0,Popularity,0.225,0.104,40
1,SVD predicted rating,0.055,0.014,40


,ratings_evaluated,mse,rmse,mean_user_mse,mean_user_rmse
0,3482,2.908,1.705,2.722,1.586


In [15]:
# WARNING: OUTPUT IS REALLY LONG

sections = ["""
<style>
.user-section { margin: 24px 0; padding: 14px; border: 1px solid #bbb; }
.user-section h2 { margin-top: 0; }
.user-section h3 { margin-bottom: 6px; }
table { border-collapse: collapse; margin-bottom: 16px; }
th, td { border: 1px solid #ccc; padding: 4px 8px; text-align: left; }
</style>
"""]

for user_id in holdout_movies_by_user:
    user_metrics = per_user_metrics[
        per_user_metrics["user_id"] == user_id
    ].sort_values("method")

    user_holdouts = heldout_movies[
        heldout_movies["user_id"] == user_id
    ].sort_values("movie_id")

    user_recommendations = top_lists[
        top_lists["user_id"] == user_id
    ].sort_values(["method", "rank"])

    sections.append(f'<section class="user-section"><h2>User {user_id}</h2>')

    sections.append("<h3>Precision and recall</h3>")
    if user_metrics.empty:
        sections.append("<p>No precision/recall scores for this user.</p>")
    else:
        sections.append(
            user_metrics.drop(columns=["user_id"], errors="ignore").to_html(
                index=False,
                na_rep="—",
                border=0,
            )
        )

    sections.append("<h3>Selected held-out movies</h3>")
    if user_holdouts.empty:
        sections.append("<p>No held-out movie rows found for this user.</p>")
    else:
        holdout_columns = [
            column for column in [
                "movie_id",
                "name",
                "predicted_rating",
                "avg_rating",
                "training_rating_count",
                "heldout_rating",
                "is_relevant",
                "meets_like_threshold",
            ]
            if column in user_holdouts.columns
        ]
        sections.append(
            user_holdouts[holdout_columns].to_html(
                index=False,
                na_rep="—",
                border=0,
            )
        )

    sections.append("<h3>Top 10 recommendations by method</h3>")
    if user_recommendations.empty:
        sections.append("<p>No recommendation rows found for this user.</p>")
    else:
        recommendation_columns = [
            column for column in [
                "method",
                "rank",
                "movie_id",
                "name",
                "predicted_rating",
                "avg_rating",
                "training_rating_count",
                "heldout_rating",
                "is_relevant",
                "meets_like_threshold",
            ]
            if column in user_recommendations.columns
        ]
        sections.append(
            user_recommendations[recommendation_columns].to_html(
                index=False,
                na_rep="—",
                border=0,
            )
        )

    sections.append("</section>")

# display(HTML("\n".join(sections)))

In [ ]:
import numpy as np
import pandas as pd

def get_debiased_recommendations(user_id, model, data, movie_stats, alpha=0.5, top_k=10):
    """
    Re-ranks recommendations by penalizing movies with high review counts.

    Parameters
   -------
    user_id : int
    model : surprise.SVD
    data : pd.DataFrame
        Training ratings with columns 'user_id', 'movie_id', 'rating'.
    movie_stats : pd.DataFrame
        Per-movie metadata with columns 'movie_id', 'name',
        'avg_rating', 'rating_count'.
    alpha : float
        Strength of the popularity penalty.
    top_k : int

    Returns
   ----
    pd.DataFrame : top_k debiased recommendations.
    """
    rated_movies = set(data.loc[data["user_id"] == user_id, "movie_id"])
    candidates = movie_stats[~movie_stats["movie_id"].isin(rated_movies)].copy()

    candidates["predicted_rating"] = [
        model.predict(user_id, mid).est for mid in candidates["movie_id"]
    ]

    log_counts = np.log1p(candidates["rating_count"])
    max_log = log_counts.max() if len(log_counts) else 1.0
    candidates["popularity_penalty"] = alpha * (log_counts / max_log) * 10.0
    candidates["debiased_score"] = (
        candidates["predicted_rating"] - candidates["popularity_penalty"]
    )

    recs = (
        candidates.sort_values(
            ["debiased_score", "movie_id"], ascending=[False, True]
        )
        .head(top_k)
        .reset_index(drop=True)
    )

    return recs[
        ["movie_id", "name", "predicted_rating", "rating_count",
         "popularity_penalty", "debiased_score", "avg_rating"]
    ]


alpha = 0.45 # mess with this. higher value = higher penalty
uid = 538143 # played around with a specific user tested above
debiased_list = get_debiased_recommendations(
    uid, model, data, movie_stats, alpha=alpha, top_k=10
)
print(debiased_list)

   movie_id                                 name  predicted_rating  \
0    637752                            8-46-2020          9.644496   
1    481169                     when-they-see-us          9.846744   
2    328580                        lemonade-2016          9.705415   
3     47133                           buffalo-66          9.559064   
4    524409                   and-then-we-danced          9.599637   
5    629266  the-history-of-the-seattle-mariners          9.084722   
6   1161806           the-seed-of-the-sacred-fig          9.078959   
7     51271                       a-few-good-men          9.366577   
8     50713              the-empire-strikes-back          9.823741   
9    476325                 house-of-hummingbird          9.007671   

   rating_count  popularity_penalty  debiased_score  avg_rating  
0            40            1.722247        7.922250    8.800000  
1            87            2.076459        7.770286    9.103448  
2            74            2.00

In [ ]:
# compare list to default

def compare_popularity(original_df, debiased_df, top_k=10,
                       baseline_sort_col="predicted_rating",
                       count_col="rating_count"):
    """
    Compare average popularity (review counts) of the top-K items under a
    baseline ranking vs a debiased ranking.

    original_df : full candidate pool
    debiased_df : output of get_debiased_recommendations
    """
    baseline_top = (
        original_df.sort_values(baseline_sort_col, ascending=False)
        .head(top_k)
    )
    debiased_top = debiased_df.head(top_k)

    base_mean = baseline_top[count_col].mean()
    deb_mean = debiased_top[count_col].mean()

    print(f"Top {top_k} Baseline (mean {count_col})")
    print(baseline_top[[ "name", baseline_sort_col, count_col ]].to_string(index=False))
    print(f"\nMean: {base_mean:.1f}")

    print(f"\nTop {top_k} Debiased (mean {count_col})")
    print(debiased_top[[ "name", "predicted_rating", count_col,
                         "debiased_score" ]].to_string(index=False))
    print(f"\nMean: {deb_mean:.1f}")

    if base_mean == 0:
        print("\nBaseline mean is 0; cannot compute reduction.")
        return

    reduction = 1 - (deb_mean / base_mean)
    print(f"\nPopularity Reduction: {reduction:.2%}")

debiased_full_list = get_debiased_recommendations(
    uid, model, data, movie_stats, alpha=alpha, top_k=10
)
compare_popularity(candidates, debiased_full_list, top_k=10)

Top 10 Baseline (mean rating_count)
                       name  predicted_rating  rating_count
all-too-well-the-short-film         10.000000           339
 taylor-swift-the-eras-tour          9.974788           340
           when-they-see-us          9.846744            87
    the-empire-strikes-back          9.823741           204
                  inception          9.786956           381
              lemonade-2016          9.705415            74
                     amelie          9.646976           166
                  8-46-2020          9.644496            40
             before-sunrise          9.630342           224
         and-then-we-danced          9.599637            89

Mean: 194.4

Top 10 Debiased (mean rating_count)
                               name  predicted_rating  rating_count  debiased_score
                      primal-2019-1          8.736314            33      -40.821972
the-history-of-the-seattle-mariners          9.084722            34      -40.880945
  